# Session 03 — Drawing and Shapes on Images

**Goal:** build visual system output on live video. Run this locally where OpenCV can access a webcam.


## Setup


In [ ]:
%pip install opencv-python
import cv2
import time


## Draw the live interface


In [ ]:
# Hold an orange object in view. These HSV limits identify it.
ORANGE_LOW = (4, 120, 90)
ORANGE_HIGH = (24, 255, 255)
ROI_WIDTH = 360
ROI_HEIGHT = 240
READY_TOLERANCE = 24  # pixels from ROI centre.

camera = cv2.VideoCapture(0)
if not camera.isOpened():
    raise RuntimeError("No webcam available. Run locally and allow camera access.")

while True:
    ok, frame = camera.read()
    if not ok:
        break
    h, w = frame.shape[:2]
    x1, y1 = (w-ROI_WIDTH)//2, (h-ROI_HEIGHT)//2
    x2, y2 = x1+ROI_WIDTH, y1+ROI_HEIGHT
    roi_centre = (x1 + ROI_WIDTH // 2, y1 + ROI_HEIGHT // 2)
    cv2.rectangle(frame, (x1,y1), (x2,y2), (21,138,194), 3)
    cv2.drawMarker(frame, roi_centre, (246,138,69), cv2.MARKER_CROSS, 20, 2)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, ORANGE_LOW, ORANGE_HIGH)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    status = 'ADJUST — show orange object'
    colour = (246,138,69)
    if contours:
        contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(contour) > 700:
            moments = cv2.moments(contour)
            if moments['m00']:
                cx, cy = int(moments['m10']/moments['m00']), int(moments['m01']/moments['m00'])
                distance = ((cx-roi_centre[0])**2 + (cy-roi_centre[1])**2)**.5
                ready = x1 <= cx <= x2 and y1 <= cy <= y2 and distance <= READY_TOLERANCE
                status = 'READY — capture frame' if ready else 'ADJUST — centre orange object'
                colour = (21,138,194) if ready else (246,138,69)
                cv2.circle(frame, (cx,cy), 8, colour, -1)
                cv2.line(frame, roi_centre, (cx,cy), colour, 2)
    cv2.putText(frame, status, (18,35), cv2.FONT_HERSHEY_SIMPLEX, .65, colour, 2)
    cv2.putText(frame, f"tolerance: {READY_TOLERANCE}px", (18,h-18), cv2.FONT_HERSHEY_SIMPLEX, .6, (255,255,255), 2)
    cv2.imshow("Alignment gate — press q to exit", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
camera.release()
cv2.destroyAllWindows()


## Evidence

Record the overlay purpose, ROI, displayed measurement, and the decision it helps a user make.
